# Results Tables & Charts
Reads `res/results.csv` and produces LaTeX tables (mean ± std across folds) and timing bar charts.

In [ ]:
# results_tables_extended_v1
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

# Load every available results CSV and concatenate. The Q1/Q2 scripts write
# per-task files; Q3 writes the legacy results.csv. Newer error metrics live
# in dedicated *_error_*.csv files. All share the same schema, including
# the ``tabpfn_version`` column added when both v2.6 and v3 were enabled.
RES_DIR = Path("../res")
# Result-file tags of the EAJ Referee 1 re-run (--results-tag): frequency with
# exposure as a TabPFN feature, severity with the 1_plus_log target. Set a tag
# to "" to read the original untagged files instead. Tagged and untagged files
# are never mixed: the dedup below would silently overwrite old with new.
FREQ_TAG = "expo"
SEV_TAG = "log1p"

def _tagged(stem: str, tag: str) -> str:
    return f"{stem}_{tag}.csv" if tag else f"{stem}.csv"

_CSV_NAMES = [
    "results.csv",
    _tagged("results_severity", SEV_TAG),
    _tagged("results_frequency", FREQ_TAG),
    _tagged("results_error_severity", SEV_TAG),
    _tagged("results_error_frequency", FREQ_TAG),
    "results_fretelematic.csv",
]
_frames = []
for name in _CSV_NAMES:
    p = RES_DIR / name
    if p.exists():
        _frames.append(pd.read_csv(p))
        print(f"loaded  {name}")
    else:
        print(f"missing {name}")
if not _frames:
    raise FileNotFoundError(
        f"No results CSVs found in {RES_DIR.resolve()}. "
        "Run the experiment scripts first."
    )
df = pd.concat(_frames, ignore_index=True)

# Older CSV writes (before the tabpfn_version column existed) lack the
# column entirely; coerce missing / NaN values to "" so non-TabPFN rows do
# not introduce NaN-keyed groups in any aggregation downstream.
if "tabpfn_version" not in df.columns:
    df["tabpfn_version"] = ""
df["tabpfn_version"] = df["tabpfn_version"].fillna("").astype(str)

# Deduplicate including tabpfn_version so v2.6 and v3 rows for the same
# (experiment, dataset, model, task, fold, metric) are kept as separate rows.
df = df.drop_duplicates(
    subset=["experiment_id", "dataset", "model", "task", "fold", "metric",
            "tabpfn_version"],
    keep="last",
)

# Synthesize a display label that fuses ``model`` + ``tabpfn_version`` so
# every downstream groupby / filter / plot can key on a single string.
# Non-TabPFN rows (empty tabpfn_version) keep their original model name.
df["model_display"] = df.apply(
    lambda r: f"{r['model']}_{r['tabpfn_version']}" if r["tabpfn_version"] else r["model"],
    axis=1,
)

# Per-fold rows only (exclude pooled aggregate)
folds = df[df["fold"] != "pooled"].copy()
folds["fold"] = folds["fold"].astype(int)

# Consistent display-model order. Each TabPFN size appears once per
# version so v2.6 and v3 are listed side-by-side.
_TABPFN_SIZES = ["", "2000", "5000", "10000", "50000", "100000", "full"]
_TABPFN_VERSIONS = ["v2_6", "v3"]

def _tabpfn_label(size: str, version: str) -> str:
    base = "tabpfn" if size == "" else f"tabpfn_{size}"
    return f"{base}_{version}"

MODEL_ORDER = ["glm", "xgboost"] + [
    _tabpfn_label(s, v) for s in _TABPFN_SIZES for v in _TABPFN_VERSIONS
]

def mean_std_table(data, metric, index_col="model_display"):
    """Pivot mean±std LaTeX strings, rows ordered by MODEL_ORDER."""
    sub = data[data["metric"] == metric]
    stats = (
        sub.groupby(["dataset", index_col])["value"]
        .agg(["mean", "std"])
        .reset_index()
    )
    stats["mean_std"] = stats.apply(
        lambda r: f"{r['mean']:.4f} $\\pm$ {r['std']:.4f}", axis=1
    )
    table = stats.pivot(index=index_col, columns="dataset", values="mean_std")
    order = [m for m in MODEL_ORDER if m in table.index]
    return table.loc[order]


# Paper figures (cells 9, 13, 15) are drawn at the EAJ text width and included
# with width=\linewidth, so these font sizes are the printed sizes.
TEXTWIDTH_IN = 372 / 72.27  # EAJ \textwidth = 372pt
# Page sizes of the paper figures. They are saved without bbox_inches="tight"
# (which trims each figure differently), so every PDF is exactly its size;
# constrained_layout keeps labels and legend inside. FIG_SIZE is for the
# two-row figures (frequency, severity). The one-row timing figure is shorter
# so its panels match theirs (~1.10 in tall): its legend, titles and x-axis
# take a fixed 0.86 in. Re-measure if FIG_SIZE or the font sizes change.
FIG_SIZE = (TEXTWIDTH_IN, 3.2)
FIG_SIZE_ONE_ROW = (TEXTWIDTH_IN, 1.965)
mpl.rcParams.update({
    "font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "grid.linewidth": 0.4, "grid.color": "0.85",
    "lines.linewidth": 1.4, "lines.markersize": 4,
    "pdf.fonttype": 42,  # embed TrueType fonts
})

# GLM / XGBoost are neutral reference series; colour is kept for the two
# TabPFN versions. Keys: "glm", "xgboost", then the version tokens.
FIG_DATASETS = ["freMTPL2", "beMTPL97"]
FIG_COLORS = {"glm": "0.15", "xgboost": "0.55", "v2_6": "#2a78d6", "v3": "#eb6834"}
FIG_LABELS = {"glm": "GLM", "xgboost": "XGBoost", "v2_6": "TabPFN v2.6", "v3": "TabPFN v3"}
FIG_SIZES = [s for s in _TABPFN_SIZES if s]  # frequency context sizes
FIG_SIZE_TICKS = ["2k", "5k", "10k", "50k", "100k", "Full"]


def fold_values(dataset, task, metric, model):
    """Per-fold values of one model_display key, indexed by fold."""
    sub = folds[
        (folds["dataset"] == dataset) & (folds["task"] == task)
        & (folds["metric"] == metric) & (folds["model_display"] == model)
    ]
    return sub.set_index("fold")["value"].sort_index()


def pct_vs_glm(dataset, task, metric, model):
    """Fold-wise % difference to the GLM on the same fold: 100 (m_k / glm_k - 1).

    Empty if the model has no rows (e.g. v2.6 on the full fold).
    """
    glm = fold_values(dataset, task, metric, "glm")
    return (100 * (fold_values(dataset, task, metric, model) / glm - 1)).dropna()


def ref_line(ax, values, key, ls="-"):
    """GLM / XGBoost reference: horizontal line at the fold mean, +- 1 sd band."""
    mean, sd = values.mean(), values.std()
    ax.axhline(mean, color=FIG_COLORS[key], ls=ls, lw=1.0, label=FIG_LABELS[key], zorder=1)
    ax.axhspan(mean - sd, mean + sd, color=FIG_COLORS[key], alpha=0.15, lw=0, zorder=0)

## 1. Deviance table — Q1 Severity (Gamma deviance, lower is better)

In [ ]:
sev = folds[folds["task"] == "sev"]
table_sev = mean_std_table(sev, "gamma_deviance")
print(table_sev.to_latex(
    escape=False,
    caption="Gamma deviance (mean $\\pm$ std over 5 folds) for severity models.",
    label="tab:severity"
))

## 2. Deviance table — Q2 Frequency (Poisson deviance, lower is better)

In [ ]:
freq = folds[folds["task"] == "freq"]
table_freq = mean_std_table(freq, "poisson_deviance")
print(table_freq.to_latex(
    escape=False,
    caption="Poisson deviance (mean $\\pm$ std over 5 folds) for frequency models.",
    label="tab:frequency"
))

## 3. Combined frequency + severity table

In [ ]:
def mean_std_series(data, metric):
    """Return a Series indexed by model_display with mean±std strings."""
    sub = data[data["metric"] == metric]
    stats = sub.groupby("model_display")["value"].agg(["mean", "std"])
    return stats.apply(lambda r: f"{r['mean']:.3f} $\\pm$ {r['std']:.3f}", axis=1)

datasets = ["freMTPL2", "beMTPL97"]
task_metric = [("freq", "poisson_deviance"), ("sev", "gamma_deviance")]

cols = {}
for dataset in datasets:
    sub_ds = folds[folds["dataset"] == dataset]
    for task, metric in task_metric:
        sub_task = sub_ds[sub_ds["task"] == task]
        cols[(dataset, task)] = mean_std_series(sub_task, metric)

combined = pd.DataFrame(cols)
combined.columns = pd.MultiIndex.from_tuples(combined.columns, names=["Dataset", "Task"])

# Consistent row order (display labels)
order = [m for m in MODEL_ORDER if m in combined.index]
combined = combined.loc[order]

print(combined.to_latex(
    escape=False,
    caption="Poisson deviance (frequency) and Gamma deviance (severity), mean $\\pm$ std over 5 folds.",
    label="tab:combined"
))

## 4. Timing — fit + predict time vs TabPFN context size (frequency)
Mean ± 1 sd over the five folds, log scale. GLM and XGBoost do not depend on context size, so they are horizontal lines with a ± 1 sd band. Severity timing is not plotted. Saved to `res/timing_barchart.pdf` (file name kept for `main.tex`).

In [ ]:
# Fit + predict time vs TabPFN context size (frequency task), log scale:
# the values span 0.1 s to several minutes, so on a linear axis the GLM /
# XGBoost bars vanish. Fold mean +- 1 sd: GLM / XGBoost as a line with a band,
# TabPFN as error bars (mostly narrower than the marker).
fig, axes = plt.subplots(
    1, len(FIG_DATASETS), figsize=FIG_SIZE_ONE_ROW,
    sharey=True, squeeze=False, constrained_layout=True,
)
x = np.arange(len(FIG_SIZES))
for ax, dataset in zip(axes[0], FIG_DATASETS):
    for model, ls in (("glm", "-"), ("xgboost", "--")):
        ref_line(ax, fold_values(dataset, "freq", "fit_predict_seconds", model), model, ls)
    for version in _TABPFN_VERSIONS:
        xs, means, sds = [], [], []
        for i, size in enumerate(FIG_SIZES):
            secs = fold_values(dataset, "freq", "fit_predict_seconds",
                               _tabpfn_label(size, version))
            if not secs.empty:  # v2.6 has no full-fold run
                xs.append(i)
                means.append(secs.mean())
                sds.append(secs.std())
        ax.errorbar(xs, means, yerr=sds, color=FIG_COLORS[version], marker="o",
                    ms=3.5, capsize=0, elinewidth=0.8, label=FIG_LABELS[version])
    ax.set_yscale("log")
    ax.grid(axis="y")
    ax.set_xticks(x, FIG_SIZE_TICKS)
    ax.set_xlabel("TabPFN context size")
    ax.set_title(dataset, fontweight="bold")
axes[0][0].set_ylabel("Time per fold (s)")  # "fit + predict" is in the caption

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=4, frameon=False)
fig.savefig("../res/timing_barchart.pdf")
plt.show()
print("Saved to res/timing_barchart.pdf")

## 5. Pooled deviance table (single aggregate score)

In [ ]:
pooled = df[df["fold"] == "pooled"].copy()
pooled_dev = pooled[pooled["metric"].isin(["gamma_deviance", "poisson_deviance"])]

table_pooled = pooled_dev.pivot_table(index="model_display", columns=["dataset", "task"], values="value")
order = [m for m in MODEL_ORDER if m in table_pooled.index]
table_pooled = table_pooled.loc[order]

print(table_pooled.round(4).to_latex(
    caption="Pooled out-of-fold deviance.",
    label="tab:pooled"
))

## 6. Frequency figure — % vs GLM by TabPFN context size
Exposure-weighted RMSE (rate scale) and Poisson deviance as the fold-wise % difference to the GLM, $100\,(m_k/\mathrm{GLM}_k - 1)$. Pairing on the fold removes the shared fold difficulty, which otherwise swamps the gaps between models. GLM and XGBoost are horizontal lines with a ± 1 sd band (the GLM band is empty by construction). Saved to `res/per_fold_freq.pdf` (file name kept for `main.tex`).

In [ ]:
# Frequency, relative to the GLM, against TabPFN context size.
# TabPFN: mean +- 1 sd over folds, faint dots = individual folds.
# GLM and XGBoost do not depend on context size: horizontal line at the fold
# mean with a +- 1 sd band (GLM is 0 on every fold by construction, so no band).
FREQ_METRICS = [
    ("exposure_weighted_rmse_rate", "Exposure-weighted RMSE"),
    ("poisson_deviance",            "Poisson deviance"),
]
VERSION_DODGE = 0.12  # x-offset between v2.6 and v3 at the same context size

fig, axes = plt.subplots(
    len(FREQ_METRICS), len(FIG_DATASETS), figsize=FIG_SIZE,
    sharex=True, squeeze=False, constrained_layout=True,
)
x = np.arange(len(FIG_SIZES))
for col, dataset in enumerate(FIG_DATASETS):
    for row, (metric, ylabel) in enumerate(FREQ_METRICS):
        ax = axes[row][col]
        for model, ls in (("glm", "-"), ("xgboost", "--")):
            ref_line(ax, pct_vs_glm(dataset, "freq", metric, model), model, ls)
        for k, version in enumerate(_TABPFN_VERSIONS):
            offset = (k - 0.5) * VERSION_DODGE
            xs, means, sds = [], [], []
            for i, size in enumerate(FIG_SIZES):
                pct = pct_vs_glm(dataset, "freq", metric, _tabpfn_label(size, version))
                if pct.empty:  # v2.6 has no full-fold run
                    continue
                ax.scatter(np.full(len(pct), i + offset), pct, s=6, lw=0,
                           color=FIG_COLORS[version], alpha=0.35, zorder=2)
                xs.append(i + offset)
                means.append(pct.mean())
                sds.append(pct.std())
            ax.errorbar(xs, means, yerr=sds, color=FIG_COLORS[version], marker="o",
                        ms=3.5, capsize=0, elinewidth=0.8,
                        label=FIG_LABELS[version], zorder=3)
        ax.grid(axis="y")
        if row == 0:
            ax.set_title(dataset, fontweight="bold")
        if col == 0:
            ax.set_ylabel(f"{ylabel}\n(% vs GLM)")
        if row == len(FREQ_METRICS) - 1:
            ax.set_xticks(x, FIG_SIZE_TICKS)
            ax.set_xlabel("TabPFN context size")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=4, frameon=False)
fig.savefig("../res/per_fold_freq.pdf")
plt.show()
print("Saved to res/per_fold_freq.pdf")

## 7. Severity figure — % vs GLM
Unweighted RMSE and Gamma deviance as the fold-wise % difference to the GLM. Mean ± 1 sd over folds; faint dots are individual folds. Saved to `res/per_fold_sev.pdf` (file name kept for `main.tex`).

In [ ]:
# Severity, relative to the GLM: fold-wise % difference, one row per model.
# Mean +- 1 sd over folds, faint dots = individual folds.
SEV_METRICS = [("rmse", "RMSE"), ("gamma_deviance", "Gamma deviance")]
# (model_display, FIG_COLORS / FIG_LABELS key), top to bottom.
SEV_MODELS = [("glm", "glm"), ("xgboost", "xgboost")] + [
    (_tabpfn_label("", v), v) for v in _TABPFN_VERSIONS
]

fig, axes = plt.subplots(
    len(SEV_METRICS), len(FIG_DATASETS), figsize=FIG_SIZE,
    sharey=True, squeeze=False, constrained_layout=True,
)
for col, dataset in enumerate(FIG_DATASETS):
    for row, (metric, xlabel) in enumerate(SEV_METRICS):
        ax = axes[row][col]
        for i, (model, key) in enumerate(SEV_MODELS):
            pct = pct_vs_glm(dataset, "sev", metric, model)
            y = len(SEV_MODELS) - 1 - i  # first model on top
            ax.scatter(pct, np.full(len(pct), y), s=10, lw=0,
                       color=FIG_COLORS[key], alpha=0.4)
            ax.errorbar(pct.mean(), y, xerr=pct.std(), color=FIG_COLORS[key],
                        marker="o", ms=4, capsize=0, elinewidth=1.0)
        ax.axvline(0, color=FIG_COLORS["glm"], lw=0.6)
        ax.grid(axis="x")
        ax.set_yticks(range(len(SEV_MODELS)),
                      [FIG_LABELS[key] for _, key in SEV_MODELS][::-1])
        ax.set_xlabel(f"{xlabel} (% vs GLM)")
        if row == 0:
            ax.set_title(dataset, fontweight="bold")

fig.savefig("../res/per_fold_sev.pdf")
plt.show()
print("Saved to res/per_fold_sev.pdf")

## 8. Per-fold severity Pearson / Spearman correlations
Only severity tasks contribute (correlations not tracked for frequency).

In [ ]:
# Per-fold severity correlations — saved to res/per_fold_correlation_lines.pdf
# Line styles moved here from the old per-fold cell (only user left).
_SIZE_STYLES = {
    "":        {"color": "#2A9D8F", "marker": "^"},
    "2000":    {"color": "#E9C46A", "marker": "o"},
    "5000":    {"color": "#F4A261", "marker": "s"},
    "10000":   {"color": "#E76F51", "marker": "^"},
    "50000":   {"color": "#9B5DE5", "marker": "D"},
    "100000":  {"color": "#264653", "marker": "P"},
    "full":    {"color": "#1D3557", "marker": "X"},
}
MODEL_STYLES = {
    "glm":     {"color": "#E63946", "marker": "o", "ls": "-"},
    "xgboost": {"color": "#457B9D", "marker": "s", "ls": "-"},
}
for _size, _base in _SIZE_STYLES.items():
    _base_label = "tabpfn" if _size == "" else f"tabpfn_{_size}"
    for _version, _ls in (("v2_6", "--"), ("v3", "-")):
        MODEL_STYLES[f"{_base_label}_{_version}"] = {**_base, "ls": _ls}

CORR_METRICS = [("pearson_corr", "Pearson"), ("spearman_corr", "Spearman")]
datasets = ["freMTPL2", "beMTPL97"]

fig_width_in  = 7.2 * 1.2
fig_height_in = 3.4 * 1.2

fig, axes = plt.subplots(
    len(datasets), len(CORR_METRICS),
    figsize=(fig_width_in, fig_height_in),
    squeeze=False,
    constrained_layout=True,
)

handles_labels = {}
for row, dataset in enumerate(datasets):
    for col, (metric, label) in enumerate(CORR_METRICS):
        ax = axes[row][col]
        sub = folds[
            (folds["dataset"] == dataset)
            & (folds["task"] == "sev")
            & (folds["metric"] == metric)
        ]
        models_present = [m for m in MODEL_ORDER if m in sub["model_display"].values]
        for model in models_present:
            md = sub[sub["model_display"] == model].sort_values("fold")
            style = MODEL_STYLES.get(model, {"color": "grey", "marker": "o", "ls": "-"})
            line, = ax.plot(
                md["fold"] + 1, md["value"],
                color=style["color"], marker=style["marker"], ls=style["ls"],
                linewidth=1.2, markersize=3, label=model, alpha=0.8,
            )
            handles_labels[model] = line
        ax.set_title(f"{dataset} — {label}")
        ax.set_xlabel("Fold")
        ax.set_ylabel(label)
        if not sub.empty:
            ax.set_xticks(range(1, sub["fold"].nunique() + 1))
        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
        ax.axhline(0, color="black", linewidth=0.5, alpha=0.5)

ordered_handles = [handles_labels[m] for m in MODEL_ORDER if m in handles_labels]
ordered_labels  = [m for m in MODEL_ORDER if m in handles_labels]
fig.legend(
    ordered_handles, ordered_labels,
    loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.08),
    title="Model", frameon=True,
)
fig.suptitle("Severity — prediction/actual correlation per fold", fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig("../res/per_fold_correlation_lines.pdf", bbox_inches="tight")
plt.show()
print("Saved to res/per_fold_correlation_lines.pdf")

## 9. Paper table — RMSE + deviance per task per dataset
Lower is better; **bold** denotes best per (dataset, task, metric).

In [ ]:
# Combined RMSE + deviance table in the paper's LaTeX format.
# Columns: dataset blocks (freMTPL2, beMTPL97) -> [freq RMSE, freq Pois Dev, sev RMSE, sev Gam Dev].

PAPER_DATASETS = ["freMTPL2", "beMTPL97"]
# Layout: (task, metric, decimal_places). RMSE for severity is shown as integers.
PAPER_COLUMNS = [
    ("freq", "exposure_weighted_rmse_rate", 3),
    ("freq", "poisson_deviance",            3),
    ("sev",  "rmse",                        0),
    ("sev",  "gamma_deviance",              3),
]
# Display label  →  model_display key. v2.6 and v3 are shown as separate rows.
PAPER_MODELS = [
    ("GLM",                  "glm"),
    ("XGBoost",              "xgboost"),
    ("TabPFN (v2.6)",        "tabpfn_v2_6"),
    ("TabPFN (v3)",          "tabpfn_v3"),
    ("TabPFN-2000 (v2.6)",   "tabpfn_2000_v2_6"),
    ("TabPFN-2000 (v3)",     "tabpfn_2000_v3"),
    ("TabPFN-5000 (v2.6)",   "tabpfn_5000_v2_6"),
    ("TabPFN-5000 (v3)",     "tabpfn_5000_v3"),
    ("TabPFN-10000 (v2.6)",  "tabpfn_10000_v2_6"),
    ("TabPFN-10000 (v3)",    "tabpfn_10000_v3"),
    ("TabPFN-50000 (v2.6)",  "tabpfn_50000_v2_6"),
    ("TabPFN-50000 (v3)",    "tabpfn_50000_v3"),
    ("TabPFN-100000 (v2.6)", "tabpfn_100000_v2_6"),
    ("TabPFN-100000 (v3)",   "tabpfn_100000_v3"),
    ("TabPFN-full (v2.6)",   "tabpfn_full_v2_6"),
    ("TabPFN-full (v3)",     "tabpfn_full_v3"),
]


def _agg(dataset: str, model: str, task: str, metric: str):
    # Return (mean, std) for one cell, or (None, None) if no rows.
    sub = folds[
        (folds["dataset"]       == dataset)
        & (folds["model_display"] == model)
        & (folds["task"]        == task)
        & (folds["metric"]      == metric)
    ]["value"]
    if sub.empty:
        return None, None
    return float(sub.mean()), float(sub.std(ddof=1))


def _fmt_cell(mean, std, decimals, best):
    if mean is None:
        return "---"
    if decimals == 0:
        body = f"{mean:.0f} \\pm {std:.0f}"
    else:
        body = f"{mean:.{decimals}f} \\pm {std:.{decimals}f}"
    return f"$\\mathbf{{{body}}}$" if best else f"${body}$"


# Pre-compute every cell so we know which mean is the column-wise minimum.
matrix = {}  # (dataset, model, task, metric) -> (mean, std)
for dataset in PAPER_DATASETS:
    for _, model in PAPER_MODELS:
        for task, metric, _ in PAPER_COLUMNS:
            matrix[(dataset, model, task, metric)] = _agg(dataset, model, task, metric)

# Best per (dataset, task, metric): smallest mean among models with data.
best_keys = set()
for dataset in PAPER_DATASETS:
    for task, metric, _ in PAPER_COLUMNS:
        candidates = [
            (m, matrix[(dataset, m, task, metric)][0])
            for _, m in PAPER_MODELS
            if matrix[(dataset, m, task, metric)][0] is not None
        ]
        if candidates:
            best_model = min(candidates, key=lambda x: x[1])[0]
            best_keys.add((dataset, best_model, task, metric))


def _row(label: str, model: str, dataset: str) -> str:
    cells = []
    for task, metric, dec in PAPER_COLUMNS:
        mean, std = matrix[(dataset, model, task, metric)]
        cells.append(_fmt_cell(mean, std, dec, (dataset, model, task, metric) in best_keys))
    # Paper layout pairs freq cells and sev cells with double-spaced wrapping.
    return (
        f"{label}\n"
        f"  & {cells[0]} & {cells[1]}\n"
        f"  & {cells[2]} & {cells[3]} \\\\"
    )


lines = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Performance metrics across datasets and tasks (mean $\pm$ std over 5 folds).",
    r"Frequency: exposure-weighted RMSE and Poisson deviance.",
    r"Severity: unweighted RMSE and Gamma deviance.",
    r"Lower is better; \textbf{bold} denotes best per metric per task.}",
    r"\label{tab:results}",
    r"\begin{tabular}{lcccc}",
    r"\toprule",
    r"& \multicolumn{2}{c}{Frequency} & \multicolumn{2}{c}{Severity} \\",
    r"\cmidrule(lr){2-3} \cmidrule(lr){4-5}",
    r"Model & RMSE & Pois.\ Dev. & RMSE & Gam.\ Dev. \\",
]
for di, dataset in enumerate(PAPER_DATASETS):
    lines += [r"\midrule", rf"\multicolumn{{5}}{{l}}{{\textit{{{dataset}}}}} \\",
              r"\midrule"]
    for ri, (label, model) in enumerate(PAPER_MODELS):
        # Visual break before the first subsample-size row.
        if model == "tabpfn_2000_v2_6":
            lines.append(r"\addlinespace[2pt]")
        lines.append(_row(label, model, dataset))
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

latex_table = "\n".join(lines)
print(latex_table)
Path("../res/results_table.tex").write_text(latex_table + "\n")
print("\nAlso saved to res/results_table.tex")

## 10. Paper table — updated version
Frequency and severity in **separate** tables (EAJ Referee 2, minor comment 1), so no cell is empty. Datasets side by side; in the frequency table the TabPFN rows are grouped under one sub-heading per version and labelled by context size; lower is better; **bold** denotes best per (dataset, metric). Writes `res/results_table_freq.tex` and `res/results_table_sev.tex`.

While `TABPFN_FREQ_CORRECTED = False`, TabPFN frequency cells carry a $\dagger$: they are still the pre-correction numbers (EAJ Referee 1). Flip it once the VSC re-run lands.

In [ ]:
# Paper tables, split by task (EAJ Referee 2, minor comment 1): frequency and
# severity are reported in separate tables so no cell is left empty. Datasets
# sit side by side as column blocks; each table keeps only the models that
# were evaluated on that task. Frequency TabPFN runs are grouped under one
# sub-heading per version, each row labelled by its context size only.
# Needs only `folds` / `MODEL_ORDER` / `_tabpfn_label` from cell 1.

SPLIT_DATASETS = ["freMTPL2", "beMTPL97"]

# The TabPFN frequency rows in res/ are the pre-correction (rate-target)
# numbers until the exposure-corrected VSC re-run lands (EAJ Referee 1).
# While False, those cells carry a dagger and the caption flags them.
# Flip to True once res/ holds the corrected v2.6 and v3 runs.
TABPFN_FREQ_CORRECTED = False

# Per table: metrics as (metric, column header, decimal places).
SPLIT_TABLES = {
    "freq": {
        "metrics": [
            ("exposure_weighted_rmse_rate", "RMSE",         3),
            ("poisson_deviance",            r"Pois.\ Dev.", 3),
        ],
        "caption": (
            r"Frequency performance: exposure-weighted RMSE and Poisson deviance "
            r"(mean $\pm$ std over 5 folds). "
            r"TabPFN uses $n$ policies of the training fold as context; "
            r"GLM and XGBoost are fitted on the full training fold. "
            r"v2.6 has no full-fold row: the training fold exceeds its 100,000-row "
            r"pretraining limit. "
            r"Lower is better; \textbf{bold} denotes best per metric per dataset."
        ),
        "label": "tab:results_freq",
        "path": RES_DIR / "results_table_freq.tex",
    },
    "sev": {
        "metrics": [
            ("rmse",           "RMSE",         0),
            ("gamma_deviance", r"Gam.\ Dev.",  3),
        ],
        "caption": (
            r"Severity performance (mean $\pm$ std over 5 folds): "
            r"unweighted RMSE and Gamma deviance. "
            r"TabPFN uses the entire training fold as context. "
            r"Lower is better; \textbf{bold} denotes best per metric per dataset."
        ),
        "label": "tab:results_sev",
        "path": RES_DIR / "results_table_sev.tex",
    },
}


def _paper_label(model: str) -> str:
    """glm -> GLM, tabpfn_v3 -> TabPFN (v3), tabpfn_2000_v2_6 -> TabPFN-2000 (v2.6)."""
    if model == "glm":
        return "GLM"
    if model == "xgboost":
        return "XGBoost"
    rest = model[len("tabpfn_"):]
    size, version = ("", rest) if rest.startswith("v") else rest.split("_", 1)
    version = version.replace("_", ".")
    return f"TabPFN ({version})" if size == "" else f"TabPFN-{size} ({version})"


def _context_label(size: str) -> str:
    """2000 -> 2,000, full -> Full fold."""
    return f"{int(size):,}" if size.isdigit() else "Full fold"


def build_split_table(task: str) -> str:
    spec = SPLIT_TABLES[task]
    metrics = spec["metrics"]
    task_rows = folds[folds["task"] == task]

    # mean / std per (dataset, model, metric) in one pass.
    stats = (
        task_rows[task_rows["metric"].isin([m for m, _, _ in metrics])]
        .groupby(["dataset", "model_display", "metric"])["value"]
        .agg(["mean", "std"])
    )

    # Rows: every model with at least one cell for this task, in MODEL_ORDER.
    present = set(stats.index.get_level_values("model_display"))
    models = [m for m in MODEL_ORDER if m in present]

    # Best (lowest mean) per (dataset, metric), as full (dataset, model, metric) keys.
    best = set(stats["mean"].groupby(level=["dataset", "metric"]).idxmin())

    def cell(dataset, model, metric, decimals):
        key = (dataset, model, metric)
        if key not in stats.index:
            print(f"WARNING: no {task} rows for {key}; cell left as ---")
            return "---"
        mean, std = stats.loc[key, "mean"], stats.loc[key, "std"]
        body = f"{mean:.{decimals}f} \\pm {std:.{decimals}f}"
        if key in best:
            body = f"\\mathbf{{{body}}}"
        if task == "freq" and model.startswith("tabpfn") and not TABPFN_FREQ_CORRECTED:
            body += "^{\\dagger}"
        return f"${body}$"

    caption = spec["caption"]
    if task == "freq" and not TABPFN_FREQ_CORRECTED:
        caption += (r" $^{\dagger}$Pre-correction TabPFN values (rate target);"
                    r" to be replaced by the exposure-corrected re-run.")

    n_cols = 1 + len(SPLIT_DATASETS) * len(metrics)
    header_blocks = " & ".join(
        rf"\multicolumn{{{len(metrics)}}}{{c}}{{{d}}}" for d in SPLIT_DATASETS
    )
    cmidrules = " ".join(
        rf"\cmidrule(lr){{{2 + i * len(metrics)}-{1 + (i + 1) * len(metrics)}}}"
        for i in range(len(SPLIT_DATASETS))
    )
    metric_headers = " & ".join([h for _, h, _ in metrics] * len(SPLIT_DATASETS))

    lines = [
        r"\begin{table}",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{spec['label']}}}",
        rf"\begin{{tabular}}{{l{'c' * (n_cols - 1)}}}",
        r"\toprule",
        rf"& {header_blocks} \\",
        cmidrules,
        rf"Model & {metric_headers} \\",
        r"\midrule",
    ]

    def row(label, model):
        cells = [
            cell(d, model, m, dec) for d in SPLIT_DATASETS for m, _, dec in metrics
        ]
        return f"{label} & " + " & ".join(cells) + r" \\"

    # Baselines first. TabPFN runs without a context size (severity) follow
    # as plain rows; context-size runs (frequency) get one sub-heading per
    # version, with rows labelled by context size only.
    unsized = {_tabpfn_label("", v) for v in _TABPFN_VERSIONS}
    lines += [row(_paper_label(m), m) for m in models if not m.startswith("tabpfn")]
    plain = [m for m in models if m in unsized]
    if plain:
        lines.append(r"\addlinespace[2pt]")
        lines += [row(_paper_label(m), m) for m in plain]
    for version in _TABPFN_VERSIONS:
        sizes = [s for s in _TABPFN_SIZES if s and _tabpfn_label(s, version) in present]
        if not sizes:
            continue
        lines += [
            r"\addlinespace[4pt]",
            rf"\multicolumn{{{n_cols}}}{{l}}{{\textit{{TabPFN {version.replace('_', '.')}}} (context size)}} \\",
        ]
        lines += [row(rf"\quad {_context_label(s)}", _tabpfn_label(s, version)) for s in sizes]
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


for task in ("freq", "sev"):
    latex_table = build_split_table(task)
    print(latex_table, end="\n\n")
    SPLIT_TABLES[task]["path"].write_text(latex_table + "\n")
    print(f"Saved to {SPLIT_TABLES[task]['path']}\n")